In [1]:
!pip install pettingzoo[mpe]
!pip install torch numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.5/852.5 kB 33.1 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import wandb
import random
import numpy as np
import imageio
from pettingzoo.classic import tictactoe_v3

In [3]:
env = tictactoe_v3.env(render_mode=None)

In [ ]:
def wandb_runs():
  wandb.login(key="")
  run = wandb.init(
    entity=" ,
    project="tic-tac-toe",
    name = "tic-tac-toe-2-actor-runs"
    )
  return run

In [5]:
MAX_EPISODES = 20_000
EPOCHS = 3
EVAL_LOOPS = 7
PPO_EPSILON = 0.12
REWARD_DECAY = 0.99
EVAL_STEPS = 100
GAE_LAMBDA = 0.95
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
class PolicyNetwork(nn.Module):

  def __init__(self, ):
    super().__init__()

    self.convolution_seq = nn.Sequential(
        nn.Conv2d(2, 16, kernel_size=3, stride=1),
        nn.ReLU(),
    )

    self.policy_head = nn.Sequential(
        nn.Linear(16, 64),
        nn.ReLU(),
        nn.Linear(64, 9)
    )

  def forward(self, x):
    x = self.convolution_seq(x)
    x = x.view(x.size(0), -1)
    x = self.policy_head(x)
    return x

  def get_action(self, x, action_mask, action=None):

    logits = self.forward(x)
    logits = logits.masked_fill(action_mask == 0, -1e9)
    dist = torch.distributions.Categorical(logits=logits)
    chosen_action = dist.sample() if action is None else action.squeeze(-1)
    log_probs = dist.log_prob(chosen_action)
    entropy = dist.entropy()
    return chosen_action, log_probs, entropy

In [7]:
class ValueNetwork(nn.Module):

  def __init__(self, ):
    super().__init__()

    self.convolution_seq = nn.Sequential(
        nn.Conv2d(2, 16, kernel_size=3, stride=1),
        nn.ReLU(),
    )

    self.policy_head = nn.Sequential(
        nn.Linear(16, 64),
        nn.ReLU(),
        nn.Linear(64, 1)
    )

  def forward(self, x):
    x = self.convolution_seq(x)
    x = x.view(x.size(0), -1)
    x = self.policy_head(x)
    return x

In [13]:
actornetwork1 = PolicyNetwork().to(DEVICE)
actornetwork2 = PolicyNetwork().to(DEVICE)
criticnetwork1 = ValueNetwork().to(DEVICE)
criticnetwork2 = ValueNetwork().to(DEVICE)

actor_optimizer1 = torch.optim.Adam(actornetwork1.parameters(), lr=LEARNING_RATE)
actor_optimizer2 = torch.optim.Adam(actornetwork2.parameters(), lr=LEARNING_RATE)
critic_optimizer1 = torch.optim.Adam(criticnetwork1.parameters(), lr=LEARNING_RATE)
critic_optimizer2 = torch.optim.Adam(criticnetwork2.parameters(), lr=LEARNING_RATE)

In [9]:
def evaluation_loop(record=False, record_episode=0):

    eval_env = tictactoe_v3.env(render_mode="rgb_array" if record else "human")
    eval_agents = eval_env.possible_agents
    rewards = {agent: [] for agent in eval_agents}

    for ep in range(EVAL_LOOPS):
        eval_env.reset()
        frames = []

        for agent in eval_env.agent_iter():
            state, reward, terminated, truncated, _ = eval_env.last()
            observation = state['observation']
            action_mask = state['action_mask']
            tensor_observation = torch.from_numpy(observation).float().to(DEVICE).permute(2, 0, 1)
            tensor_action_mask = torch.from_numpy(action_mask).float().to(DEVICE)

            with torch.no_grad():

                if agent == eval_agents[0]:
                    actornetwork = actornetwork1
                else:
                    actornetwork = actornetwork2

                logits = actornetwork(tensor_observation.unsqueeze(0))
                logits = logits.masked_fill(tensor_action_mask == 0, -1e9)
                action = torch.argmax(logits).item()

            rewards[agent].append(reward)

            # capture frame after every step
            if record and ep == record_episode:
                frame = eval_env.render()  # [H, W, 3] numpy array
                frames.append(frame)

            if terminated or truncated:
                eval_env.step(None)
            else:
                eval_env.step(action)

        # save video for the chosen episode
        if record and ep == record_episode and frames:
            imageio.mimsave("eval_episode.gif", frames, fps=2)
            # runs.log({"evaluation": wandb.Video("eval_episode.gif", fps=2, format="gif")})

    reward_agent1 = sum(rewards[eval_agents[0]]) / EVAL_LOOPS
    reward_agent2 = sum(rewards[eval_agents[1]]) / EVAL_LOOPS
    eval_env.close()

    return reward_agent1, reward_agent2

In [10]:
agents = env.possible_agents

# runs = wandb_runs()

global_steps = 0
alternate_step = 0

In [12]:
agents

['player_1', 'player_2']

In [ ]:
evaluation_loop(record=True, record_episode=0)

In [ ]:
for step in range(MAX_EPISODES):

  env.reset()

  buffer = {
      agent: {
          "observations": [],
          "actions": [],
          "mask": [],
          "log_probs": [],
          "rewards": [],
          "dones": [],
          "gae": []
        }
      for agent in env.agents
      }

  for agent in env.agent_iter():
    state, reward, terminated, truncated, _ = env.last()

    observation = state['observation']    # type: ignore
    # 3x3x2 dim matrix where first two dim represents the row-col and the later dim represents the {x} {o} in binary format

    action_mask = state['action_mask']   # type: ignore
    # 1 means where we can put the agent in board and 0 means we are not allowed.

    tensor_observation = torch.from_numpy(observation).float().to(DEVICE).permute(2, 0, 1) #converting from (H, W, C) to (C, H, W) for CNN
    tensor_action_mask = torch.from_numpy(action_mask).float().to(DEVICE)

    with torch.no_grad():
      if agent == agents[0]:
        actornetwork = actornetwork1
      else:
        actornetwork = actornetwork2

      action, log_probs, entropy = actornetwork.get_action(tensor_observation.unsqueeze(0), tensor_action_mask.unsqueeze(0), None)


    reward = torch.tensor(reward, dtype=torch.float32).to(DEVICE)
    terminated = torch.tensor(terminated, dtype=torch.float).to(DEVICE)

    buffer[agent]['observations'].append(tensor_observation.detach())
    buffer[agent]['actions'].append(action.detach())
    buffer[agent]['log_probs'].append(log_probs.detach())
    buffer[agent]['rewards'].append(reward.detach())
    buffer[agent]['dones'].append(terminated.detach())
    buffer[agent]['mask'].append(tensor_action_mask.detach())

    if terminated:
      env.step(None)
    else:
      env.step(action.item())
    global_steps += 1

  for agent in agents:
      buffer[agent]['observations'] = torch.stack(buffer[agent]['observations'])
      buffer[agent]['actions'] = torch.stack(buffer[agent]['actions'])
      buffer[agent]['log_probs'] = torch.stack(buffer[agent]['log_probs'])
      buffer[agent]['rewards'] = torch.stack(buffer[agent]['rewards'])
      buffer[agent]['dones'] = torch.stack(buffer[agent]['dones'])
      buffer[agent]['mask'] = torch.stack(buffer[agent]['mask'])
      buffer[agent]['next_observations'] = torch.zeros_like(buffer[agent]['observations'])
      buffer[agent]['next_observations'][:-1] = buffer[agent]['observations'][1:]

      if agent == agents[0]:
          criticnetwork = criticnetwork1
      else:
          criticnetwork = criticnetwork2

      with torch.no_grad():

        vt_1 = criticnetwork(buffer[agent]['next_observations']).squeeze(-1)
        vt__ = criticnetwork(buffer[agent]['observations']).squeeze(-1)

      delta = (buffer[agent]['rewards'] + REWARD_DECAY * vt_1  *(1 - buffer[agent]['dones'])) - vt__          # type: ignore

      var_delt = 0
      idx = len(delta) - 1
      for delt in reversed(delta):
        var_delt = delt + (GAE_LAMBDA* REWARD_DECAY* var_delt)*(1 - buffer[agent]['dones'][idx])
        buffer[agent]['gae'].insert(0, var_delt)
        idx -= 1

      buffer[agent]['gae'] = torch.stack(buffer[agent]['gae'])

      buffer[agent]['gae'] = (buffer[agent]['gae'] - buffer[agent]['gae'].mean()) / (buffer[agent]['gae'].std() + 1e-5)

  all_states_agent1 = buffer[agents[0]]['observations']
  all_states_agent2 = buffer[agents[1]]['observations']

  all_mask_agent1 = buffer[agents[0]]['mask']
  all_mask_agent2 =  buffer[agents[1]]['mask']

  all_actions_agent1 = buffer[agents[0]]['actions']
  all_actions_agent2 = buffer[agents[1]]['actions']

  old_log_probs_agent1 = buffer[agents[0]]['log_probs']
  old_log_probs_agent2 = buffer[agents[1]]['log_probs']

  all_gae_agent1 = buffer[agents[0]]['gae']
  all_gae_agent2 = buffer[agents[1]]['gae']

  all_rewards1 = buffer[agents[0]]['rewards']
  all_rewards2 = buffer[agents[1]]['rewards']

  with torch.no_grad():
    all_returns1 = all_gae_agent1 + criticnetwork1(all_states_agent1).squeeze(-1)
    all_returns2 = all_gae_agent2 + criticnetwork2(all_states_agent2).squeeze(-1)

  log_actorloss1 = 0
  log_actorloss2 = 0
  log_criticloss1 = 0
  log_criticloss2 = 0
  log_step = 0
  log_entropy1 = 0
  log_entropy2 = 0
  log_kl_div1 = 0
  log_kl_div2 = 0

  for epoch in range(EPOCHS):

      critic_returns1 = criticnetwork1(all_states_agent1).squeeze(-1)
      critic_returns2 = criticnetwork2(all_states_agent2).squeeze(-1)

      critic_error1 = torch.nn.functional.mse_loss(critic_returns1, all_returns1.detach())
      critic_error2 = torch.nn.functional.mse_loss(critic_returns2, all_returns2.detach())

      _, new_log_probs1, entropy1 = actornetwork1.get_action(all_states_agent1, all_mask_agent1, all_actions_agent1)
      _, new_log_probs2, entropy2 = actornetwork2.get_action(all_states_agent2, all_mask_agent2, all_actions_agent2)

      kl_div1 = (old_log_probs_agent1 - new_log_probs1).mean()
      kl_div2 = (old_log_probs_agent2 - new_log_probs2).mean()

      ratio1 = torch.exp(new_log_probs1 - old_log_probs_agent1)
      ratio2 = torch.exp(new_log_probs2 - old_log_probs_agent2)

      ppo_loss1 = - torch.min(ratio1 * all_gae_agent1, torch.clamp(ratio1, 1 - PPO_EPSILON, 1 + PPO_EPSILON) * all_gae_agent1).mean()
      ppo_loss2 = - torch.min(ratio2 * all_gae_agent2, torch.clamp(ratio2, 1 - PPO_EPSILON, 1 + PPO_EPSILON) * all_gae_agent2).mean()

      critic_optimizer1.zero_grad()
      critic_optimizer2.zero_grad()
      critic_error1.backward()
      critic_error2.backward()
      torch.nn.utils.clip_grad_norm_(criticnetwork1.parameters(), 0.5)
      torch.nn.utils.clip_grad_norm_(criticnetwork2.parameters(), 0.5)
      critic_optimizer1.step()
      critic_optimizer2.step()

      actor_optimizer1.zero_grad()
      actor_optimizer2.zero_grad()
      ppo_loss1.backward()
      ppo_loss2.backward()
      torch.nn.utils.clip_grad_norm_(actornetwork1.parameters(), 0.5)
      torch.nn.utils.clip_grad_norm_(actornetwork2.parameters(), 0.5)
      actor_optimizer1.step()
      actor_optimizer2.step()

      log_actorloss1 += ppo_loss1.item()
      log_actorloss2 += ppo_loss2.item()
      log_entropy1 += entropy1.mean().item()
      log_entropy2 += entropy2.mean().item()
      log_criticloss1 += critic_error1.item()
      log_criticloss2 += critic_error2.item()
      log_kl_div1 += kl_div1.item()
      log_kl_div2 += kl_div2.item()
      log_step += 1

  if alternate_step%EVAL_STEPS==0:
      reward_agent1, reward_agent2 = evaluation_loop()
      runs.log({
          "eval-agent-1": reward_agent1,
          "eval-agent-2": reward_agent2
          }, step=global_steps)

  runs.log(
      {
          "actor-loss1": log_actorloss1/log_step,
          "actor-loss2": log_actorloss2/log_step,
          "critic-loss1": log_criticloss1/log_step,
          "critic-loss2": log_criticloss2/log_step,
          "entropy1": log_entropy1/log_step,
          "entropy2": log_entropy2/log_step,
          "kl-div1": log_kl_div1/log_step,
          "kl-div2": log_kl_div2/log_step,
          "agent-1-reward": all_rewards1.mean().item(),
          "agent-2-reward": all_rewards2.mean().item()
      },
      step=global_steps
  )

  alternate_step += 1